# 🚀 BLEnD CultureRAG - Complete Pipeline

## ⚠️ IMPORTANT: Run Cells in Order

**Active cells (run these):**
- Cells 1-9: Complete RRF + Context-aware RAG pipeline
- Cell 10-12: Benchmarking (optional)

**Deprecated cells (DO NOT RUN):**
- Cells 13-21: Old implementations, kept for reference only
- ⚠️ **Cell 13 filters to English-only (12 rows) - DO NOT USE FOR SUBMISSION**

## 📊 Expected Results
- Full dataset: **148 questions** across all languages
- Outputs: `predictions_rrf_context.tsv` (submit this) + `retrieval_logs.tsv` (debug)

---

In [7]:
!pip install -q unsloth
!pip install -qU transformers sentence-transformers faiss-cpu
!pip install -q wikipedia-api beautifulsoup4 requests
print("✅ Installation complete")


✅ Installation complete


In [8]:
import pandas as pd
import torch
import re
import os
import requests
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig  # FIXED
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import time

print("✅ Libraries imported")


✅ Libraries imported


In [ ]:
# ============================================================================
# TASK 1.1: spaCy Entity Extractor (NER-based)
# ============================================================================
!pip install -q spacy
!python -m spacy download en_core_web_sm

import spacy
import re
from collections import defaultdict

nlp = spacy.load("en_core_web_sm")

NER_KEEP = {"GPE", "LOC", "PERSON", "ORG", "EVENT", "WORK_OF_ART"}

DEFAULT_BLACKLIST = {
    "What", "Which", "Who", "Where", "When", "Why", "How",
    "The", "A", "An", "In", "On", "At", "Of", "For", "From",
    "Option", "Options"
}

ACRONYM_RE = re.compile(r"\b[A-Z]{2,}\b")  # ONLY all-caps fallback (e.g., HDB, UK)

def extract_entities_spacy(row, nlp, blacklist=None):
    """Extract entities using spaCy NER + acronym fallback"""
    blacklist = set(DEFAULT_BLACKLIST if blacklist is None else blacklist)

    text = " ".join([
        str(row.get("question", "")),
        str(row.get("option_A", "")),
        str(row.get("option_B", "")),
        str(row.get("option_C", "")),
        str(row.get("option_D", "")),
    ])

    doc = nlp(text)

    ents = set()
    for ent in doc.ents:
        if ent.label_ in NER_KEEP:
            cand = ent.text.strip()
            if cand and cand not in blacklist:
                ents.add(cand)

    # Regex fallback ONLY for acronyms (avoid TitleCase regex entirely)
    for ac in ACRONYM_RE.findall(text):
        if ac not in blacklist:
            ents.add(ac)

    # Optional: drop ultra-short non-acronyms
    ents = {e for e in ents if (len(e) >= 3 or e.isupper())}

    country = str(row.get("id", "")).split("-")[1][:2] if "id" in row else None
    return {"id": row.get("id", None), "country": country, "entities": sorted(ents)}

# Apply to all questions
df = pd.read_csv('track_2_mcq_input.tsv', sep='\t')
entity_data = [extract_entities_spacy(row, nlp) for row in df.to_dict("records")]

# See what we extracted
print(f"Total questions: {len(entity_data)}")
print(f"Example entities: {entity_data[0]}")

# Count unique countries
countries = set([d['country'] for d in entity_data if d['country']])
print(f"Countries covered: {len(countries)}")
print(f"Countries: {sorted(countries)}")
print(f"\n✅ Upgraded to spaCy NER-based entity extraction")
# Checkpoint: You should see ~30 unique countries and cleaner entity lists per question


In [ ]:
# ============================================================================
# Persistent Wikipedia Cache (Disk-backed)
# ============================================================================
import os
import pickle
from pathlib import Path

# Use Kaggle working dir if available; otherwise current dir
CACHE_FILE = "/kaggle/working/wiki_cache.pkl"
if not Path(CACHE_FILE).parent.exists():
    CACHE_FILE = "wiki_cache.pkl"


def load_wiki_cache():
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, "rb") as f:
                cache = pickle.load(f)
            print(f"📦 Loaded {len(cache)} cached pages from disk")
            return cache
        except Exception as e:
            print(f"⚠️ Could not load cache: {e}")
    return {}


def save_wiki_cache(cache):
    try:
        with open(CACHE_FILE, "wb") as f:
            pickle.dump(cache, f)
        print(f"💾 Saved cache: {len(cache)} pages -> {CACHE_FILE}")
    except Exception as e:
        print(f"⚠️ Could not save cache: {e}")


# Global cache shared with scraper
wiki_cache = load_wiki_cache()


In [ ]:
import requests
from bs4 import BeautifulSoup
import time
from tqdm.auto import tqdm
import json
import pickle
import os
from pathlib import Path

class EntityWikipediaScraper:
    def __init__(self, country_sources):
        self.country_sources = country_sources
        self.cache = wiki_cache  # Use global disk-backed cache

    def search_wikipedia(self, entity):
        """Search Wikipedia for entity and return top article"""
        url = "https://en.wikipedia.org/w/api.php"
        params = {
            'action': 'opensearch',
            'search': entity,
            'limit': 1,
            'format': 'json'
        }
        try:
            response = requests.get(url, params=params, timeout=5)
            results = response.json()
            if len(results) > 3 and len(results[3]) > 0:
                return results[3][0]  # Return URL
        except:
            pass
        return None
    
    def scrape_page(self, page_title):
        """Scrape a Wikipedia page and return paragraphs (with disk cache)"""
        # Check disk cache first
        if page_title in self.cache:
            return self.cache[page_title]
        
        url = f"https://en.wikipedia.org/wiki/{page_title}"
        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')
            
            content = soup.find('div', {'id': 'mw-content-text'})
            if not content:
                return []
            
            paragraphs = []
            for p in content.find_all('p'):
                text = p.get_text().strip()
                text = ' '.join(text.split())  # Normalize whitespace
                
                # Quality filters
                if len(text) < 100:  # Too short
                    continue
                if text.count('[') > 5:  # Too many citations
                    continue
                    
                paragraphs.append(text)
            
            # Cache and periodic save
            self.cache[page_title] = paragraphs
            if len(self.cache) % 10 == 0:
                save_wiki_cache(self.cache)
            time.sleep(0.5)  # Rate limiting
            return paragraphs
            
        except Exception as e:
            print(f"Error scraping {page_title}: {e}")
            return []
    
    def build_kb(self, entity_data):
        """Build knowledge base from entity data"""
        kb_chunks = []
        
        # 1. Get country-specific base pages
        print("Scraping country-specific pages...")
        countries_seen = set()
        for item in tqdm(entity_data):
            country = item['country']
            if country in countries_seen:
                continue
            countries_seen.add(country)
            
            if country in self.country_sources:
                for page_title in self.country_sources[country]:
                    paragraphs = self.scrape_page(page_title)
                    for p in paragraphs:
                        kb_chunks.append({
                            'text': p,
                            'country': country,
                            'source': page_title,
                            'type': 'base'
                        })
        
        print(f"Scraped {len(kb_chunks)} chunks from base pages")
        
        # 2. Get entity-specific pages (ALL questions with rate limiting)
        print("Scraping entity-specific pages...")
        entity_count = 0
        max_entities = 200  # Reasonable limit with caching
        
        for item in tqdm(entity_data):  # Process ALL questions
            if entity_count >= max_entities:
                break
                
            for entity in item['entities'][:2]:  # Top 2 entities per question
                if len(entity) < 4:  # Skip short strings
                    continue
                
                url = self.search_wikipedia(entity)
                if url:
                    page_title = url.split('/')[-1]
                    
                    # Check cache to avoid duplicate scraping
                    if page_title in self.cache:
                        paragraphs = self.cache[page_title]
                    else:
                        paragraphs = self.scrape_page(page_title)
                    
                    for p in paragraphs[:2]:  # Top 2 paragraphs per entity
                        kb_chunks.append({
                            'text': p,
                            'country': item['country'],
                            'source': page_title,
                            'entity': entity,
                            'type': 'entity'
                        })
                    entity_count += 1
                    
                    if entity_count >= max_entities:
                        break
        
        print(f"Total KB chunks: {len(kb_chunks)}")
        print(f"✅ Disk cache now has {len(self.cache)} pages")
        return kb_chunks

# Load config and build KB
with open('country_sources.json', 'r') as f:
    country_sources = json.load(f)

scraper = EntityWikipediaScraper(country_sources)
kb_chunks = scraper.build_kb(entity_data)

# Save KB to file
with open('kb_chunks.pkl', 'wb') as f:
    pickle.dump(kb_chunks, f)

# Final save of cache
save_wiki_cache(wiki_cache)

print(f"\n✅ KB built with {len(kb_chunks)} chunks")
print(f"   - Base pages: {sum(1 for c in kb_chunks if c['type']=='base')}")
print(f"   - Entity pages: {sum(1 for c in kb_chunks if c['type']=='entity')}")
print(f"   - Wikipedia cache: {len(scraper.cache)} pages cached to disk")
# Checkpoint: You should have 400-800 chunks total

In [ ]:
!pip install -q rank-bm25 nltk sentence-transformers faiss-cpu
import nltk
nltk.download('punkt')
print("✅ Dependencies installed")


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
import nltk
import pickle

# Load KB
with open('kb_chunks.pkl', 'rb') as f:
    kb_chunks = pickle.load(f)

kb_texts = [chunk['text'] for chunk in kb_chunks]

print("Building FAISS index...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(kb_texts, show_progress_bar=True, convert_to_numpy=True)

# Normalize for cosine similarity
faiss.normalize_L2(embeddings)

# Build FAISS index
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)  # Inner product = cosine after normalization
faiss_index.add(embeddings)

print(f"✅ FAISS index built: {faiss_index.ntotal} vectors")

# Build BM25 index
print("Building BM25 index...")
tokenized_kb = [nltk.word_tokenize(text.lower()) for text in kb_texts]
bm25 = BM25Okapi(tokenized_kb)

print(f"✅ BM25 index built")


In [ ]:
# ============================================================================
# Hybrid Retrieval with Reciprocal Rank Fusion (RRF)
# ============================================================================

from collections import defaultdict
import numpy as np
import nltk
import faiss


def hybrid_retrieve_rrf(question, country_filter=None, top_k=5, candidate_k=50, k_rrf=60):
    """
    Hybrid retrieval with Reciprocal Rank Fusion (RRF)
    RRF score: 1 / (k_rrf + rank + 1) — scale-invariant
    """
    # Step 1: Country filtering
    if country_filter:
        valid_indices = [i for i, c in enumerate(kb_chunks) if c['country'] == country_filter]
        if len(valid_indices) < 3:
            valid_indices = list(range(len(kb_chunks)))
    else:
        valid_indices = list(range(len(kb_chunks)))
    
    # Step 2: BM25 ranking (get top candidate_k from all, then filter)
    query_tokens = nltk.word_tokenize(question.lower())
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_ranked = np.argsort(bm25_scores)[::-1][:candidate_k * 2]
    bm25_ranked = [i for i in bm25_ranked if i in valid_indices][:candidate_k]
    
    # Step 3: FAISS ranking
    query_emb = embedder.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(query_emb)
    distances, faiss_indices = faiss_index.search(query_emb, candidate_k * 2)
    faiss_ranked = [i for i in faiss_indices[0] if i in valid_indices][:candidate_k]
    
    # Step 4: RRF fusion
    rrf_scores = defaultdict(float)
    
    for rank, idx in enumerate(bm25_ranked):
        rrf_scores[idx] += 1.0 / (k_rrf + rank + 1)
    
    for rank, idx in enumerate(faiss_ranked):
        rrf_scores[idx] += 1.0 / (k_rrf + rank + 1)
    
    # Step 5: Sort and return top-k
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    return [
        {
            'text': kb_chunks[idx]['text'],
            'country': kb_chunks[idx]['country'],
            'source': kb_chunks[idx]['source'],
            'score': score,
            'index': idx
        }
        for idx, score in sorted_results
    ]


class HybridRetrieverWrapper:
    """Thin wrapper to provide a .search(...) API expected by predict_row."""
    def search(self, query, country_filter=None, k=3):
        results = hybrid_retrieve_rrf(query, country_filter=country_filter, top_k=k)
        return [
            {
                'page_content': r['text'],
                'country': r['country'],
                'source': r['source'],
                'score': r['score'],
                'index': r['index'],
            }
            for r in results
        ]


retriever = HybridRetrieverWrapper()

print("✅ RRF hybrid retriever ready")

# Quick smoke test
_test_q = df.iloc[0]
_country = _test_q['id'].split('-')[1].split('_')[0]
_res = retriever.search(_test_q['question'], country_filter=_country, k=3)
print(f"Question: {_test_q['question'][:80]}...")
print(f"Country filter: {_country}")
for i, r in enumerate(_res):
    print(f"{i+1}. [RRF={r['score']:.4f}] [{r['source']}] {r['page_content'][:120]}...")


In [ ]:
# ============================================================================
# Constrained 1-token prediction (no post-hoc parsing)
# ============================================================================

import torch


def predict_row(row, hybrid_retriever, model, tokenizer):
    """
    Deterministic 1-token decoding. Forces the model to emit exactly one letter.
    """
    # 1) Option-aware query
    expanded_query = f"{row['question']} {row['option_A']} {row['option_B']} {row['option_C']} {row['option_D']}"

    # 2) Retrieval with optional country filter
    country_filter = row['id'].split('-')[1].split('_')[0] if '-' in row['id'] else None
    docs = hybrid_retriever.search(expanded_query, country_filter=country_filter, k=3)

    def _text_from_doc(doc):
        if hasattr(doc, 'page_content'):
            return getattr(doc, 'page_content')
        if isinstance(doc, dict) and 'page_content' in doc:
            return doc['page_content']
        if isinstance(doc, dict) and 'text' in doc:
            return doc['text']
        return str(doc)

    context_text = "\n".join([f"- {_text_from_doc(d)[:400]}" for d in docs]) if docs else "- (no context)"

    # 3) Direct prompt (no reasoning instructions)
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert on cultural knowledge. Answer the multiple-choice question using the Context.

Context:
{context_text}

Question: {row['question']}
Options:
A) {row['option_A']}
B) {row['option_B']}
C) {row['option_C']}
D) {row['option_D']}

Answer with ONLY the option letter (A, B, C, or D).
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Answer:"""

    # 4) Constrained generation (1 token)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=1,  # force single token
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 5) Decode only the newly generated token
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip().upper()

    # 6) Parse trivially
    pred = gen_text[:1] if gen_text else ""
    if pred not in ["A", "B", "C", "D"]:
        return "C"  # Safe fallback
    return pred


print("✅ predict_row updated: 1-token constrained decoding (A/B/C/D only)")


In [ ]:
# ============================================================================
# ROBUST INFERENCE WITH CHECKPOINT SAVING + SAFETY INTERLOCKS
# ============================================================================
import traceback
from tqdm.auto import tqdm
import os

EXPECTED_ROWS = 148  # dataset size guardrail


def run_experiment_safe(df, method_name, use_rag=True, checkpoint_every=10):
    """
    Safe inference with automatic checkpointing and resume.
    - Saves checkpoints to /kaggle/working/ so they persist across crashes.
    - Skips already-completed rows on resume.
    - Falls back to 'C' on error.
    - Enforces row-count and ID integrity before final save.
    """
    output_file = f"/kaggle/working/predictions_{method_name}_checkpoint.tsv"
    results = []

    # Try to resume from checkpoint
    if os.path.exists(output_file):
        try:
            existing = pd.read_csv(output_file, sep='\t', header=None, names=['id', 'prediction'])
            completed_ids = set(existing['id'].tolist())
            results.extend(existing.to_dict('records'))
            print(f"📦 Resuming {method_name}: {len(completed_ids)} already completed")
        except Exception as e:
            print(f"⚠️ Could not load checkpoint: {e}")
            completed_ids = set()
    else:
        completed_ids = set()

    for _, row in tqdm(df.iterrows(), total=len(df), desc=method_name):
        # Skip if already done
        if row['id'] in completed_ids:
            continue

        try:
            if use_rag:
                pred = predict_row(
                    row,
                    hybrid_retriever=retriever,
                    model=model,
                    tokenizer=tokenizer,
                )
            else:
                pred = "C"  # simple baseline placeholder
            results.append({'id': row['id'], 'prediction': pred})
        except Exception as e:
            print(f"\n⚠️ ERROR on {row['id']}: {e}")
            traceback.print_exc()
            results.append({'id': row['id'], 'prediction': 'C'})  # common fallback

        # Checkpoint every N new rows (not total rows)
        if len(results) % checkpoint_every == 0:
            pd.DataFrame(results).to_csv(output_file, sep='\t', index=False, header=False)

    # Safety interlocks before final save
    assert len(results) == EXPECTED_ROWS, \
        f"❌ FATAL ERROR: Generated {len(results)} rows. Expected {EXPECTED_ROWS}. Did you filter the dataframe?"
    ids = [r['id'] for r in results]
    assert len(set(ids)) == len(ids), "❌ FATAL ERROR: Duplicate IDs found. Check your loop logic."
    unique_regions = set([i.split('_')[0] for i in ids])
    print(f"✅ Success: Covered {len(unique_regions)} unique language-locales.")
    print("🛡️ Safety Checks Passed.")

    # Final save
    df_final = pd.DataFrame(results)
    final_path = f"/kaggle/working/predictions_{method_name}.tsv"
    df_final.to_csv(final_path, sep='\t', index=False, header=False)
    print(f"✅ Saved {len(results)} predictions to {final_path}")

    return results


print("Running crash-proof inference with checkpointing...\n")

experiments = {}
experiments['baseline'] = run_experiment_safe(df, 'baseline', use_rag=False)
experiments['rag_rrf_k3'] = run_experiment_safe(df, 'rag_rrf_k3', use_rag=True)
experiments['rag_rrf_k5'] = run_experiment_safe(df, 'rag_rrf_k5', use_rag=True)

print("\n" + "="*60)
print("ABLATION RESULTS")
print("="*60)
for exp_name, results in experiments.items():
    preds = [r['prediction'] for r in results]
    print(f"{exp_name:15s}: {dict(pd.Series(preds).value_counts())}")


In [ ]:
# Find cases where hybrid changed the answer
baseline_preds = {r['id']: r['prediction'] for r in experiments['baseline']}
hybrid_preds = {r['id']: r['prediction'] for r in experiments['rag_rrf_k3']}

changed = []
for qid in baseline_preds:
    if baseline_preds[qid] != hybrid_preds[qid]:
        changed.append(qid)

print(f"Hybrid changed {len(changed)} predictions")

# Manually inspect first 5
for qid in changed[:5]:
    row = df[df['id'] == qid].iloc[0]
    print(f"\n{'='*60}")
    print(f"ID: {qid}")
    print(f"Question: {row['question']}")
    print(f"A) {row['option_A']}")
    print(f"B) {row['option_B']}")
    print(f"C) {row['option_C']}")
    print(f"D) {row['option_D']}")
    print(f"Baseline: {baseline_preds[qid]}")
    print(f"Hybrid: {hybrid_preds[qid]}")
    
    # Show retrieved context (option-aware query)
    country = qid.split('-')[1].split('_')[0]
    expanded_q = " ".join([
        row['question'], row['option_A'], row['option_B'], row['option_C'], row['option_D']
    ])
    ctx = hybrid_retrieve_rrf(expanded_q, country_filter=country, top_k=2)
    print(f"\nRetrieved context:")
    for i, c in enumerate(ctx):
        print(f"{i+1}. [RRF={c['score']:.4f}] [{c['source']}] {c['text'][:200]}...")


In [ ]:
import torch

# Measure current memory usage
current_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"Current GPU memory usage: {current_mem:.2f} GB")

# Get current memory stats
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# Note: To measure FP16 baseline, run in a separate notebook:
# model_fp16 = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-3.1-8B-Instruct",
#     torch_dtype=torch.float16,
#     device_map="auto"
# )
# fp16_mem = torch.cuda.max_memory_allocated() / 1e9
# print(f"FP16 memory: {fp16_mem:.2f} GB")
# print(f"Reduction: {(fp16_mem - current_mem) / fp16_mem * 100:.1f}%")

print("\n✅ Memory benchmarking complete")
print("   For FP16 comparison, measure in separate session to avoid contamination")

In [ ]:
import time

# Benchmark retrieval only
latencies_retrieval = []
for i in range(50):
    row = df.iloc[i]
    country = row['id'].split('-')[1].split('_')[0]
    expanded_q = " ".join([
        row['question'], row['option_A'], row['option_B'], row['option_C'], row['option_D']
    ])
    
    start = time.time()
    _ = hybrid_retrieve_rrf(expanded_q, country_filter=country, top_k=3)
    latencies_retrieval.append((time.time() - start) * 1000)

print("Retrieval latency (ms):")
print(f"  Mean: {np.mean(latencies_retrieval):.1f}")
print(f"  Median: {np.median(latencies_retrieval):.1f}")
print(f"  P95: {np.percentile(latencies_retrieval, 95):.1f}")

# Benchmark end-to-end (retrieval + generation)
latencies_e2e = []
for i in range(20):  # Smaller sample (slower)
    row = df.iloc[i]
    start = time.time()
    _ = predict_row(row, hybrid_retriever=retriever, model=model, tokenizer=tokenizer)
    latencies_e2e.append((time.time() - start) * 1000)

print("\nEnd-to-end latency (ms):")
print(f"  Mean: {np.mean(latencies_e2e):.1f}")
print(f"  Median: {np.median(latencies_e2e):.1f}")
print(f"  P95: {np.percentile(latencies_e2e, 95):.1f}")


In [ ]:
# Use the path you discovered
input_path = "/kaggle/input/track-2-mcq-input-tsv/track_2_mcq_input.tsv"

df = pd.read_csv(input_path, sep='\t')
print(f"✅ Loaded {len(df)} total questions (no language filtering)")
print(f"Columns: {df.columns.tolist()}")

# Quick sample
print("\nSample row:")
print(df.iloc[0])


✅ Loaded 148 total questions
Columns: ['id', 'question', 'option_A', 'option_B', 'option_C', 'option_D']
🔍 Filtered to 12 English questions

Sample row:
id                                                  en-GB_030
question    Which of these family holiday destinations is ...
option_A                                            Ben Nevis
option_B                                    The Lake District
option_C                                            Loch Ness
option_D                                            Snowdonia
Name: 29, dtype: object
